# BriskVLA: teacher evaluation on LIBERO

Evaluates the released SmolVLA LIBERO checkpoint (`lerobot/smolvla_libero`) on the four LIBERO suites.

**How to use**
1. *Runtime → Change runtime type →* pick a **GPU**. Write down which one; latency must later be measured on a single fixed GPU type.
2. Edit the **Config** cell, then *Run all*.
3. Results go to Google Drive. If Colab disconnects, just *Run all* again: suites that already have results are skipped.

Success rate does not depend on the GPU type; latency does (not measured here).

## Config

In [ ]:
REPO_URL = "https://github.com/vyvas33/briskvla.git"
BRANCH = "briskvla"

POLICY = "lerobot/smolvla_libero"
POLICY_REVISION = None  # None = pin to the current Hub commit (recorded in run_metadata.json)

SUITES = ["libero_spatial", "libero_object", "libero_goal", "libero_10"]
N_EPISODES = 10  # per task -> 10 tasks x 10 = 100 episodes per suite
NUM_STEPS = 10   # denoising steps; set 3 for the "fewer steps" baseline
SEED = 1000      # lerobot-eval default

RUN_NAME = f"teacher_steps{NUM_STEPS}_seed{SEED}"
DRIVE_ROOT = "/content/drive/MyDrive/briskvla/eval"

## 1. GPU and Google Drive

In [ ]:
import subprocess

gpu = subprocess.run(
    ["nvidia-smi", "--query-gpu=name,memory.total,driver_version", "--format=csv,noheader"],
    capture_output=True, text=True,
).stdout.strip()
assert gpu, "No GPU found. Runtime -> Change runtime type -> GPU"
print("GPU:", gpu)

from google.colab import drive

drive.mount("/content/drive")

## 2. Install

In [ ]:
!apt-get -qq install -y libegl1 libgl1 ffmpeg > /dev/null

import os
import shutil

# The uv installer location varies; pip puts uv on PATH reliably.
UV = shutil.which("uv")
if UV is None:
    subprocess.run(["pip", "install", "-q", "uv"], check=True)
    UV = shutil.which("uv")
assert UV, "uv not found after pip install"
print("uv:", UV, subprocess.run([UV, "--version"], capture_output=True, text=True).stdout.strip())

REPO_DIR = "/content/briskvla"

if not os.path.exists(REPO_DIR):
    subprocess.run(["git", "clone", "-b", BRANCH, REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"], check=True)

subprocess.run([UV, "sync", "--locked", "--python", "3.12", "--extra", "smolvla", "--extra", "libero"], cwd=REPO_DIR, check=True)

COMMIT = subprocess.run(["git", "-C", REPO_DIR, "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip()
print("Repo commit:", COMMIT)

## 3. LIBERO assets and config

`hf-libero` asks a Y/N question with `input()` on first import if `~/.libero/config.yaml` is missing, which would hang the run.
This writes the config first, the same way the repo's `docker/Dockerfile.benchmark.libero` does.

In [ ]:
os.environ["MUJOCO_GL"] = "egl"
os.environ["PYOPENGL_PLATFORM"] = "egl"

libero_home = os.path.expanduser("~/.libero")
os.makedirs(libero_home, exist_ok=True)

setup = f'''
import importlib.util, os
from huggingface_hub import snapshot_download
snapshot_download(repo_id="lerobot/libero-assets", repo_type="dataset", local_dir="{libero_home}/assets")
libero_dir = os.path.join(os.path.dirname(importlib.util.find_spec("libero").origin), "libero")
with open("{libero_home}/config.yaml", "w") as f:
    f.write(
        f"assets: {libero_home}/assets\\n"
        f"bddl_files: {{libero_dir}}/bddl_files\\n"
        f"datasets: {{libero_dir}}/../datasets\\n"
        f"init_states: {{libero_dir}}/init_files\\n"
    )
'''
subprocess.run([UV, "run", "--no-sync", "python", "-c", setup], cwd=REPO_DIR, check=True)
print(open(f"{libero_home}/config.yaml").read())

## 4. Pin the checkpoint

In [ ]:
from huggingface_hub import HfApi, snapshot_download

POLICY_SHA = HfApi().model_info(POLICY, revision=POLICY_REVISION).sha
POLICY_DIR = snapshot_download(POLICY, revision=POLICY_SHA, local_dir=f"/content/policies/{POLICY.replace('/', '__')}@{POLICY_SHA[:8]}")
print("Policy:", POLICY, "@", POLICY_SHA)

## 5. Eval helper

In [ ]:
import json
from pathlib import Path

RUN_DIR = Path(DRIVE_ROOT) / RUN_NAME
RUN_DIR.mkdir(parents=True, exist_ok=True)

CAMERAS = '{"agentview_image": "camera1", "robot0_eye_in_hand_image": "camera2"}'


def run_eval(suite: str, n_episodes: int, out_dir: Path) -> dict | None:
    info_path = out_dir / "eval_info.json"
    if info_path.exists():
        print(f"[skip] {suite}: results already in {out_dir}")
        return json.loads(info_path.read_text())
    cmd = [
        UV, "run", "--no-sync", "lerobot-eval",
        f"--policy.path={POLICY_DIR}",
        "--policy.device=cuda",
        f"--policy.num_steps={NUM_STEPS}",
        "--policy.empty_cameras=1",
        "--env.type=libero",
        f"--env.task={suite}",
        f"--env.camera_name_mapping={CAMERAS}",
        "--eval.batch_size=1",
        f"--eval.n_episodes={n_episodes}",
        "--eval.use_async_envs=false",
        f"--seed={SEED}",
        f"--output_dir={out_dir}",
    ]
    print(" ".join(cmd))
    proc = subprocess.run(cmd, cwd=REPO_DIR)
    if proc.returncode != 0 or not info_path.exists():
        print(f"[FAILED] {suite} (exit code {proc.returncode})")
        return None
    return json.loads(info_path.read_text())

## 6. Smoke test (1 episode per task on `libero_spatial`)

Same setup as the repo's CI. Check that it finishes, then watch one video in `smoke/videos/` before the full run.

In [ ]:
smoke = run_eval("libero_spatial", 1, RUN_DIR / "smoke")
if smoke:
    o = smoke["overall"]
    print(f"smoke: {o['n_success']}/{o['n_episodes']} success, {o['eval_ep_s']:.1f} s/episode")

## 7. Full evaluation (one suite at a time, saved to Drive)

In [ ]:
torch_version = subprocess.run(
    [UV, "run", "--no-sync", "python", "-c", "import torch; print(torch.__version__)"],
    cwd=REPO_DIR, capture_output=True, text=True,
).stdout.strip()

(RUN_DIR / "run_metadata.json").write_text(json.dumps({
    "gpu": gpu,
    "torch": torch_version,
    "repo_commit": COMMIT,
    "policy": POLICY,
    "policy_sha": POLICY_SHA,
    "suites": SUITES,
    "n_episodes_per_task": N_EPISODES,
    "num_steps": NUM_STEPS,
    "seed": SEED,
}, indent=2))

results = {}
for suite in SUITES:
    results[suite] = run_eval(suite, N_EPISODES, RUN_DIR / suite)

## 8. Summary

In [ ]:
rows = []
for suite in SUITES:
    info_path = RUN_DIR / suite / "eval_info.json"
    if not info_path.exists():
        rows.append((suite, None))
        continue
    rows.append((suite, json.loads(info_path.read_text())["overall"]))

print(f"{RUN_NAME}  |  {POLICY}@{POLICY_SHA[:8]}  |  {gpu}\n")
print(f"{'suite':<16}{'success %':>10}{'n':>10}{'95% CI':>18}{'s/ep':>8}")
done = []
for suite, o in rows:
    if o is None:
        print(f"{suite:<16}{'missing':>10}")
        continue
    lo, hi = o["pc_success_ci95"]
    print(f"{suite:<16}{o['pc_success']:>10.1f}{o['n_success']:>5}/{o['n_episodes']:<4}{lo:>8.1f}-{hi:<8.1f}{o['eval_ep_s']:>7.1f}")
    done.append(o)
if len(done) == len(SUITES):
    avg = sum(o["pc_success"] for o in done) / len(done)
    print(f"{'average':<16}{avg:>10.1f}   (SmolVLA paper reports 87.3)")

(RUN_DIR / "summary.json").write_text(json.dumps({s: o for s, o in rows}, indent=2))